In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.simulation import predict_match, Matchprediction
from src.features import update_team_history, update_h2h
import matplotlib as plt
import numpy as np

In [2]:
#Load everything from earlier notebooks
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
scaler = joblib.load('scaler.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

In [5]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0,
                     "GD": 0,
                     'GF': 0,
                     "GA": 0})
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler)
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GD'] += x.home_score - x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GD'] += x.away_score - x.home_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GF'] += x.home_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GF'] += x.away_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GA'] += x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GA'] += x.home_score
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5
    
    new_away, new_home = update_elo(S, match.neutral, match.K_factor, match.home_score, match.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    
    update_team_history(match, history_dict)
    update_h2h(match, h2h_dict)

group_stage_result = group_stage_result.sort_values(['Group', 'Points', 'GD', 'GF'], ascending=[True, False, False, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results


   Group                    Team  Points  GD  GF  GA
0      A                  Mexico       7   3   6   3
1      A          Czech Republic       4   0   4   4
2      A             South Korea       3  -1   4   5
3      A            South Africa       3  -2   4   6
4      B                  Canada       9   3  13  10
5      B  Bosnia and Herzegovina       3  -1  10  11
6      B             Switzerland       3  -1   6   7
7      B                   Qatar       3  -1   5   6
8      C                 Morocco       5   1   5   4
9      C                  Brazil       4   0   4   4
10     C                Scotland       4  -1   2   3
11     C                   Haiti       3   0   6   6
12     D           United States       7   2   8   6
13     D                  Turkey       4   0   4   4
14     D               Australia       3  -1   7   8
15     D                Paraguay       3  -1   6   7
16     E                 Germany       7   8   9   1
17     E                 Ecuador       4   0  